# Phase 4 - OGB datasets under the official protocol

Runs the pipeline on OGB-sourced datasets, **structural only**, scored with **official OGB splits + metrics** (`eval_ogb.py`). Heavy lifting lives in `run_ogb.py`; this notebook only orchestrates.

- **Protocol:** train on official TRAIN only → select on **validation**, lock to `results/ogb_selection.json` (§6) → read **test once** (§7, guarded by the lock).
- **Datasets:** `ogbl_ddi` (link prediction, Hits@20; graph = training links only) · `ogbn_arxiv` (node classification, Accuracy; full citation graph, labels used for learning = training papers only). One notebook run per knob.
- **Existing routes only:** virtual graphs + deepwalk → nb2 zone, graphsage → nb3 zone, rows → `results/scoreboard.csv`.


In [1]:
"""Setup: repo root, config, knobs. Heavy lifting: experiments/run_ogb.py (ensure_virtual / embed / score / cached / table / select / report)."""
import os, sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)                     # run everything from the repo root
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import networkx as nx

from virgo import config as cfg
from virgo import graph_io
from virgo.data import make_ogb
from virgo.eval import results_io
from experiments.run_ogb import TASKS, ensure_virtual, embed, score, cached, table, select, selection, report

DATASET = "ogbl_ddi"                                # <-- pick: "ogbl_ddi" (link pred, Hits@20) | "ogbn_arxiv" (node class, Accuracy)
K = 10                                              # locked by the Phase-3 ablations
SEEDS = cfg.VG_SEEDS                                # encoder seeds 42/43/44; the build seed stays REPRO["seed"]
SIMS = cfg.VG_SIMS                                  # the seven official variants: psi / degree / centrality / original / hybrid / hybrid_degree / hybrid_centrality
ENCODERS = ["graphsage_edge", "deepwalk"]
TASK_STR = TASKS[DATASET][1]
print(f"{DATASET} | task = {TASK_STR} | K={K} | seeds {SEEDS}")

/home/m-adam/miniconda/envs/i2v/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ogbl_ddi | task = link prediction (OGB official) | K=10 | seeds [42, 43, 44]


## 1 · Get the data

`make_ogb` downloads OGB once → edgelist + `.nodes` + labels/pairs + official split. Reused when present; `data.x` never loaded (structural-only). ddi's edgelist holds **training links only**.


In [2]:
"""OGB -> project files (edgelist + .nodes + labels/pairs + official split); reuse-or-create."""
info = make_ogb.ensure_ogb(DATASET)
info


{'edge_path': 'input/ogbl_ddi_train.edgelist',
 'pairs': 'splits/ogb/ogbl_ddi_pairs.npz',
 'eval': 'ogb',
 'task': 'linkpred'}

## 2 · Load + check


In [3]:
"""One shared graph definition; check() prints the facts every downstream stage assumes."""
EDGELIST = str(cfg.DATASETS[DATASET]["edgelist"])
G = graph_io.load_graph(EDGELIST)                   # .nodes sidecar restores isolated nodes
props = graph_io.check(G, DATASET)


ogbl_ddi: 4267 nodes, 1067911 edges, max degree 2234


## 3 · Virtual graphs

Five variants, build-or-reuse; each build logs a row to `results/graph_health.csv`.


In [4]:
"""Build-or-reuse the five graph variants (top-K structural neighbors)."""
VG = {sim: ensure_virtual(G, DATASET, K, sim) for sim in SIMS}
for sim, V in VG.items():
    print(f"{sim:>10}: {V.number_of_nodes()} nodes / {V.number_of_edges()} edges")


       psi: 4267 nodes / 27042 edges
    degree: 4267 nodes / 27109 edges
centrality: 4267 nodes / 24886 edges
  original: 4267 nodes / 1067911 edges
    hybrid: 4267 nodes / 1088727 edges


## 4 · Train embeddings

Every (variant × encoder × seed), trained on the official TRAIN graph only - unsupervised, labels never touch training; reuse-or-create. GraphSAGE → nb3 zone, DeepWalk → nb2 zone. Restarted kernel ⇒ rerun §0–§4 (instant reuse).


In [5]:
"""Train-or-reuse every (variant x encoder x seed) embedding on the official TRAIN graph."""
EMB = {(sim, enc, s): embed(G, VG[sim], DATASET, K, sim, enc, s, EDGELIST)
       for sim in SIMS for enc in ENCODERS for s in SEEDS}
print(f"{len(EMB)} embeddings ready")


reuse  psi graphsage_edge seed 42 -> graphsage_edge_s42.emb
reuse  psi graphsage_edge seed 43 -> graphsage_edge_s43.emb
reuse  psi graphsage_edge seed 44 -> graphsage_edge_s44.emb
reuse  psi deepwalk seed 42 -> deepwalk_s42.emb
reuse  psi deepwalk seed 43 -> deepwalk_s43.emb
reuse  psi deepwalk seed 44 -> deepwalk_s44.emb
reuse  degree graphsage_edge seed 42 -> graphsage_edge_s42.emb
reuse  degree graphsage_edge seed 43 -> graphsage_edge_s43.emb
reuse  degree graphsage_edge seed 44 -> graphsage_edge_s44.emb
reuse  degree deepwalk seed 42 -> deepwalk_s42.emb
reuse  degree deepwalk seed 43 -> deepwalk_s43.emb
reuse  degree deepwalk seed 44 -> deepwalk_s44.emb
reuse  centrality graphsage_edge seed 42 -> graphsage_edge_s42.emb
reuse  centrality graphsage_edge seed 43 -> graphsage_edge_s43.emb
reuse  centrality graphsage_edge seed 44 -> graphsage_edge_s44.emb
reuse  centrality deepwalk seed 42 -> deepwalk_s42.emb
reuse  centrality deepwalk seed 43 -> deepwalk_s43.emb
reuse  centrality deepw

## 5 · Validation scores

The selection metric → `results/scoreboard.csv` as `valid_acc` / `valid_hits@20`.

- **ddi:** pairs are scored by a **trained decoder** (hadamard → MLP), fitted per config on **training edges only** - same decoder + seed for every variant. ~17 s per config, so this cell takes ~15 min.
- **arxiv:** the probe learns from **training-paper labels only**.


"""Score every embedding on the OFFICIAL VALIDATION split -> scoreboard rows, then the graph x encoder table."""
for sim in SIMS:
for enc in ENCODERS:
per = {}
for s in SEEDS:
for m, v in score(DATASET, str(EMB[(sim, enc, s)]), "valid", s).items():
per.setdefault(m, []).append(v)
for m, vals in per.items():
results_io.record_score(DATASET, enc, sim, K, TASK_STR, SEEDS, vals, metric=m)
print(f"{sim:>10} {enc:>14} | " + " ".join(f"{m}={np.mean(v):.4f}" for m, v in per.items()), flush=True)

table(DATASET, "valid", better=True) # one row per graph variant, one column per encoder


In [6]:
"""Score every embedding on the OFFICIAL VALIDATION split -> scoreboard rows (reuse cached rows), then the graph x encoder table."""
for sim in SIMS:
    for enc in ENCODERS:
        hit = cached(DATASET, enc, sim, K, "valid")
        if hit:                                          # deterministic scorer + reused embeddings -> reuse saved rows, skip recompute
            print(f"{sim:>10} {enc:>14} | reuse " + " ".join(f"{m}={v:.4f}" for m, v in hit.items()), flush=True)
            continue
        per = {}
        for s in SEEDS:
            for m, v in score(DATASET, str(EMB[(sim, enc, s)]), "valid", s).items():
                per.setdefault(m, []).append(v)
        for m, vals in per.items():
            results_io.record_score(DATASET, enc, sim, K, TASK_STR, SEEDS, vals, metric=m)
        print(f"{sim:>10} {enc:>14} | " + " ".join(f"{m}={np.mean(v):.4f}" for m, v in per.items()), flush=True)

table(DATASET, "valid", better=True)                # one row per graph variant, one column per encoder

       psi graphsage_edge | reuse valid_hits@20=0.0251
       psi       deepwalk | reuse valid_hits@20=0.0071
    degree graphsage_edge | reuse valid_hits@20=0.0227
    degree       deepwalk | reuse valid_hits@20=0.0071
centrality graphsage_edge | reuse valid_hits@20=0.0326
centrality       deepwalk | reuse valid_hits@20=0.0077
  original graphsage_edge | reuse valid_hits@20=0.0413
  original       deepwalk | reuse valid_hits@20=0.0772
    hybrid graphsage_edge | reuse valid_hits@20=0.0554
    hybrid       deepwalk | reuse valid_hits@20=0.1038


,GraphSAGE Hits@20,DeepWalk Hits@20,Better encoder
Graph version,,,
Ψ,0.0251,0.0071,GraphSAGE
Degree,0.0227,0.0071,GraphSAGE
Centrality,0.0326,0.0077,GraphSAGE
Original,0.0413,0.0772,DeepWalk
Hybrid,0.0554,0.1038,DeepWalk - best overall


## 6 · Selection (validation) - locks the winner

Winner picked on validation and **saved to `results/ogb_selection.json`**. §7 refuses without it; once test rows exist, re-selection refuses. Test never changes the winner.


In [7]:
"""Validation table + winner, LOCKED to results/ogb_selection.json."""
sel = select(DATASET)


ogbl_ddi already locked (2026-07-23T06:42:40): Hybrid graph + DeepWalk, Hits@20 = 0.1038 ± 0.0040   (unchanged -> /home/m-adam/identity2vec/results/ogb_selection.json)


## 7 · Final test - run ONCE, after §6 is locked

Guarded: reads the §6 lock and refuses without it. Reuses the saved embeddings; retrains nothing except the ddi decoder (deterministic, seeded, train-edges only).


In [8]:
"""ONE test read -> test_* scoreboard rows (arxiv also gets weighted/macro F1 secondaries; reuse cached rows), then the test table."""
locked = selection(DATASET)                          # the saved §6 choice; asserts the lock exists BEFORE any test score
for sim in SIMS:
    for enc in ENCODERS:
        hit = cached(DATASET, enc, sim, K, "test")
        if hit:                                          # deterministic scorer + reused embeddings -> reuse saved rows, skip recompute
            print(f"{sim:>10} {enc:>14} | reuse " + " ".join(f"{m}={v:.4f}" for m, v in hit.items()), flush=True)
            continue
        per = {}
        for s in SEEDS:
            for m, v in score(DATASET, str(EMB[(sim, enc, s)]), "test", s).items():
                per.setdefault(m, []).append(v)
        for m, vals in per.items():
            results_io.record_score(DATASET, enc, sim, K, TASK_STR, SEEDS, vals, metric=m)
        print(f"{sim:>10} {enc:>14} | " + " ".join(f"{m}={np.mean(v):.4f}" for m, v in per.items()), flush=True)

table(DATASET, "test")                               # reported, never re-selected from

       psi graphsage_edge | reuse test_hits@20=0.0423
       psi       deepwalk | reuse test_hits@20=0.0081
    degree graphsage_edge | reuse test_hits@20=0.0331
    degree       deepwalk | reuse test_hits@20=0.0080
centrality graphsage_edge | reuse test_hits@20=0.0519
centrality       deepwalk | reuse test_hits@20=0.0100
  original graphsage_edge | reuse test_hits@20=0.0173
  original       deepwalk | reuse test_hits@20=0.0378
    hybrid graphsage_edge | reuse test_hits@20=0.0344
    hybrid       deepwalk | reuse test_hits@20=0.0533


,GraphSAGE test Hits@20,DeepWalk test Hits@20
Graph,,
Ψ,0.0423,0.0081
Degree,0.0331,0.0080
Centrality,0.0519,0.0100
Original,0.0173,0.0378
Hybrid,0.0344,0.0533


## 8 · Results

Locked validation winner → its test score → both tables → the one-line conclusion. The winner is read from the §6 lock, never re-picked from the test table.


In [9]:
"""Locked winner + its test score, both tables, and the conclusion line."""
report(DATASET)

Validation winner (locked 2026-07-23T06:42:40): Hybrid graph + DeepWalk, Hits@20 = 0.1038 ± 0.0040

Validation
               GraphSAGE Hits@20  DeepWalk Hits@20           Better encoder
Graph version                                                              
Ψ                         0.0251            0.0071                GraphSAGE
Degree                    0.0227            0.0071                GraphSAGE
Centrality                0.0326            0.0077                GraphSAGE
Original                  0.0413            0.0772                 DeepWalk
Hybrid                    0.0554            0.1038  DeepWalk - best overall

Test
            GraphSAGE test Hits@20  DeepWalk test Hits@20
Graph                                                    
Ψ                           0.0423                 0.0081
Degree                      0.0331                 0.0080
Centrality                  0.0519                 0.0100
Original                    0.0173                 0.0378
Hyb